# **Trabalho Final POO - Análise de Sentimentos**
## **Disciplina: Programação Orientada a Objetos**
## **Professor: Dirson**
## **Turma: D01**

## **Alunos:**
**- Gustavo Rodrigues Ribeiro / RA:202003570** \
**- Breno Machado Barros / RA:202014607** \

# **Domínio do Negócio: E-commerce**
### Para este projeto, o domínio de negócio escolhido será o de e-commerce com foco na análise de sentimentos em avaliações de produtos. As avaliações são obtidas de um dataset público de avaliações de produtos eletrônicos, como o Amazon Product Reviews Dataset, que oferece avaliações reais em várias categorias de produtos.

### **Atenção!!**
**Segue abaixo o link para download do Dataset bruto (Electronics_5.json) utilizado no projeto:**

https://drive.google.com/uc?export=download&id=1kI0hoiWyGaJ2iy3f5RNhhJLP1Uzu_x_L

## Iniciando o PySpark

Esta célula de código instala o Spark no ambiente de execução Colab. Aqui está uma explicação passo a passo:

1. **`!apt-get install openjdk-11-jdk-headless -qq > /dev/null`**: este comando instala o OpenJDK 11 (versão headless, sem interface gráfica), que é um requisito para o Spark. O `-qq` suprime a saída e o `> /dev/null` redireciona a saída para o nada, tornando o processo mais silencioso.

2. **`!wget -q https://dlcdn.apache.org/spark/spark-3.5.2/spark-3.5.3-bin-hadoop3.tgz`**: Este comando baixa o arquivo compactado do Spark 3.5.2 (construído para o Hadoop 3) do site oficial do Apache Spark. O `-q` suprime a saída de download.

3. **`!tar xf spark-3.5.3-bin-hadoop3.tgz`**: Este comando extrai o arquivo compactado baixado do Spark, criando um diretório chamado `spark-3.5.3-bin-hadoop3`.

4. **`!pip -q install findspark`**: Este comando instala a biblioteca `findspark` usando `pip`. Findspark é uma biblioteca Python que torna mais fácil configurar o Spark em um ambiente Python, principalmente no Colab. Ela define as variáveis de ambiente necessárias para que o Spark funcione corretamente.

Após executar essas linhas, você terá o Spark instalado e pronto para ser usado em seu notebook Colab.

In [ ]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz
!tar xf spark-3.5.3-bin-hadoop3.tgz
!pip -q install findspark

Defina as variáveis de ambiente do Spark:

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.3-bin-hadoop3"

O código a seguir garante que o Spark seja configurado corretamente e esteja pronto para uso em seu ambiente Python.

* **`findspark.init()`**: executa a função `init()` do módulo `findspark`. Esta função:
    * Localiza a instalação do Spark em seu sistema.
    * Configura as variáveis de ambiente necessárias para que o Python possa interagir com o Spark. Isso permite que o driver Python (seu código Python) se comunique com o executor Spark (o código que realmente processa os dados).


In [ ]:
import findspark
findspark.init()

Depois de executar a célula anterior, você poderá importar e usar as bibliotecas Spark como `pyspark.sql.SparkSession` para criar uma sessão Spark e começar a trabalhar com dados.

**OBS: Vale lembra que esse é um código para a criação de um modelo de treinamento e teste de IA para análise de sentimento através de um dataset de avaliações de produtos, com cerca de 6000000 de reviews. Logo, é importante entender que apenas o Colab (versão gratuita) não possui recursos computacionais (GPU e RAM) suficientes para executar o modelo por completo. Assim, recomendamos a utilização da máquina local com cerca de 64gb de RAM ou uma máquina virtual como a N-highmem-64gb no Dataproc do Google Cloud Console, com o ambiente virtual Jupyter Notebook. Assim, o código irá executar sem erros de memória ou GPU.**

**OBS 2: Caso utilize uma máquina virtual como a N-highmem-64gb no Dataproc do Google Cloud Console, com o ambiente virtual Jupyter Notebook, não serão necessários os passos acima, apenas continue daqui.**

**OBS 3: No caso da Camada Bronze, o código atual, ele pode ser executado no Google Colab sem problemas, assim irá funcionar corretamente.**

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName('Trabalho Final Bronze').master("local[*]").getOrCreate()

# Caso necessário utilize spark.stop() para encerrar a sessão atual e reiniciar a SparkSession com as configurações abaixo (Caso utilize uma máquina virtual como a n2-highmem-8 no Dataproc do Google Cloud Console)
# spark.stop()
# spark = SparkSession.builder.appName('Trabalho Final Bronze').config("spark.driver.memory", "64g").config("spark.executor.memory", "64g").config("spark.executor.cores", "8").master("local[*]").getOrCreate()

print("Versão do Spark:", spark.version)

Versão do Spark: 3.5.3


## **Arquitetura Medallion: BRONZE**
## **Coleta e Ingestão de Dados**

Aqui estaremos sincronizando nossa conta no Drive ao ambiente Colab, para que os arquivos em nuvem sejam gerenciados (lidos e escritos) e manipulados diretamente no Drive.

**OBS: Caso esteja utilizando o Dataproc do Google Cloud Console, com o ambiente virtual Jupyter Notebook, você podera utilizar o Data Lake Google Cloud Storage (GCS) que está conectado a sua conta, não necessitando desse processo de sincronização com o Drive.**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!ls /content/drive

MyDrive  Shareddrives


## Arquivos Salvos em .parquet

In [ ]:
# ---------------------------
# Camada Bronze: Dados Brutos
# ---------------------------

# Carregar dados de reviews do Drive ou GCS
dataset_path = "/content/drive/MyDrive/Disciplinas-UFG/PDM/TrabalhoFinal/DatasetOriginal/Electronics_5.json"  # Mude para o diretório desejado

# Define o esquema dos dados a serem lidos no .json
df_schema = StructType([
    StructField("image", ArrayType(StringType()), True),  # Define "image" como um array de strings
    StructField("overall", FloatType(), True),
    StructField("vote", StringType(), True),
    StructField("verified", BooleanType(), True),
    StructField("reviewTime", StringType(), True),
    StructField("reviewerID", StringType(), True),
    StructField("asin", StringType(), True),
    StructField("style", MapType(StringType(), StringType()), True),
    StructField("reviewerName", StringType(), True),
    StructField("reviewText", StringType(), True),
    StructField("summary", StringType(), True),
    StructField("unixReviewTime", LongType(), True)
])

# Ler os dados diretamente do Drive inferindo o schema
reviews_df_bronze = spark.read.schema(df_schema).json(dataset_path)

# Exibir schema dos dados carregados
print()
print("Dados brutos:")
print()
reviews_df_bronze.printSchema()

# Exibir amostra dos dados carregados (5 primeiras linhas)
reviews_df_bronze.show(5)

# Salva o DataFrame como tabela da camada Bronze (formato .parquet)
# obs: Altere o caminho do arquivo para sua preferência, desde que esteja em seu Drive
reviews_df_bronze.coalesce(1).write.mode("overwrite").option("header", True).parquet("/content/drive/MyDrive/Disciplinas-UFG/PDM/TrabalhoFinal/ArquiteturaMedallion/Bronze/reviews_bronze")


Dados brutos:

root
 |-- image: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- overall: float (nullable = true)
 |-- vote: string (nullable = true)
 |-- verified: boolean (nullable = true)
 |-- reviewTime: string (nullable = true)
 |-- reviewerID: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- style: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- reviewerName: string (nullable = true)
 |-- reviewText: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- unixReviewTime: long (nullable = true)

+-----+-------+----+--------+-----------+--------------+----------+--------------------+----------------+--------------------+--------------------+--------------+
|image|overall|vote|verified| reviewTime|    reviewerID|      asin|               style|    reviewerName|          reviewText|             summary|unixReviewTime|
+-----+-------+----+--------+-----------+--------------+-